# NLP Experiment 4 – Named Entity Recognition with spaCy
**Runtime:** Python 3 (CPU). Runtime → Run all.

**NER** finds names of things in text and labels them: PERSON, ORG (organisation), GPE (country/city/state),
DATE, MONEY, … (a) uses spaCy's **pre-trained** model; (b) **customises** it for medical terms.

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy
from spacy import displacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")      # if this fails: Runtime -> Restart session, run again
print(nlp.pipe_names)

## (a) Pre-trained NER on a news article

In [ ]:
article = """Apple Inc. announced on 1 September 2026 that it will invest $5 billion in a new research
centre in California. Chief executive Tim Cook said the company will create more than 10,000 jobs over the
next five years. Microsoft and Google are also expanding in the United States, while Infosys opened a new
campus in Pune, India, led by Salil Parekh."""

doc = nlp(article)

for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:10} {spacy.explain(ent.label_)}")

In [ ]:
# entities as a table + how many of each type
table = pd.DataFrame([(ent.text, ent.label_) for ent in doc.ents], columns=["entity", "label"])
print(table["label"].value_counts())
table

In [ ]:
displacy.render(doc, style="ent", jupyter=True)    # highlighted entities

## (b) Customise for domain-specific (medical) entities
The pre-trained model doesn't know DISEASE or MEDICATION. Two ways to add them:
1. **EntityRuler** (rule-based): give spaCy a list of patterns. Simple and exact.
2. **Train** the NER on labelled examples (statistical, can generalise).

In [ ]:
medical_text = "The patient was diagnosed with diabetes and hypertension and was prescribed Metformin and Amlodipine by Dr. Mehta in Mumbai."

print("Before customisation:")
for ent in nlp(medical_text).ents:
    print(" ", ent.text, "->", ent.label_)

### Option 1: EntityRuler

In [ ]:
ruler = nlp.add_pipe("entity_ruler", before="ner")    # rules run first, before the statistical NER

patterns = [
    {"label": "DISEASE", "pattern": "diabetes"},
    {"label": "DISEASE", "pattern": "hypertension"},
    {"label": "DISEASE", "pattern": "asthma"},
    {"label": "DISEASE", "pattern": "cancer"},
    {"label": "MEDICATION", "pattern": "Metformin"},
    {"label": "MEDICATION", "pattern": "Amlodipine"},
    {"label": "MEDICATION", "pattern": "Insulin"},
    {"label": "MEDICATION", "pattern": "Salbutamol"},
]
ruler.add_patterns(patterns)

doc2 = nlp(medical_text)
print("After customisation:")
for ent in doc2.ents:
    print(" ", ent.text, "->", ent.label_)

displacy.render(doc2, style="ent", jupyter=True)

### Option 2: train a NER model on labelled examples
Training data = sentence + list of (start_char, end_char, label). We **compute** the positions with
`text.find(word)` so they are always correct (wrong positions = entities silently ignored).

In [ ]:
import random
from spacy.training import Example

sentences = [
    ("The patient was diagnosed with diabetes and prescribed Metformin.", [("diabetes", "DISEASE"), ("Metformin", "MEDICATION")]),
    ("The doctor prescribed Insulin to treat diabetes.", [("Insulin", "MEDICATION"), ("diabetes", "DISEASE")]),
    ("Hypertension can be treated using Amlodipine.", [("Hypertension", "DISEASE"), ("Amlodipine", "MEDICATION")]),
    ("The patient has asthma and uses Salbutamol.", [("asthma", "DISEASE"), ("Salbutamol", "MEDICATION")]),
    ("Doctors treat cancer with chemotherapy.", [("cancer", "DISEASE"), ("chemotherapy", "TREATMENT")]),
    ("She takes Amlodipine for hypertension.", [("Amlodipine", "MEDICATION"), ("hypertension", "DISEASE")]),
    ("Asthma attacks were reduced by Salbutamol.", [("Asthma", "DISEASE"), ("Salbutamol", "MEDICATION")]),
    ("He started chemotherapy for lung cancer.", [("chemotherapy", "TREATMENT"), ("cancer", "DISEASE")]),
]

TRAIN_DATA = []
for text, words in sentences:
    entities = []
    for word, label in words:
        start = text.find(word)
        entities.append((start, start + len(word), label))
    TRAIN_DATA.append((text, {"entities": entities}))

print(TRAIN_DATA[0])

In [ ]:
custom = spacy.blank("en")            # empty English pipeline
ner = custom.add_pipe("ner")
for label in ["DISEASE", "MEDICATION", "TREATMENT"]:
    ner.add_label(label)

optimizer = custom.initialize()
random.seed(1)
for epoch in range(30):
    random.shuffle(TRAIN_DATA)
    losses = {}
    for text, annotations in TRAIN_DATA:
        example = Example.from_dict(custom.make_doc(text), annotations)
        custom.update([example], drop=0.2, sgd=optimizer, losses=losses)
    if epoch % 5 == 0:
        print("epoch", epoch, "loss", round(losses["ner"], 3))

In [ ]:
test = "The patient has diabetes and was given Insulin, then started chemotherapy."
for ent in custom(test).ents:
    print(ent.text, "->", ent.label_)